# Pipeline Output - Database Inspector

This notebook displays the contents of the SQLite database produced by the pipeline.

In [67]:
import sys
import os
from pathlib import Path

# 1. Locate the project root and put its src/ on sys.path.
_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
_nb_dir = Path(os.path.abspath('')).resolve()
if _nb_dir not in _candidates:
    _candidates += [_nb_dir, *_nb_dir.parents]
_project_root = next(
    (c for c in _candidates if (c / 'src' / 'financial_sentiment').is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError("Could not find project src/ directory.")
if str(_project_root / 'src') not in sys.path:
    sys.path.insert(0, str(_project_root / 'src'))

# 2. Drop any stale cached version of the package (e.g. from a previous
# Jupyter kernel that ran against an older src/) so the import picks up
# the current modules on disk.
for _mod in [m for m in sys.modules if m == 'financial_sentiment' or m.startswith('financial_sentiment.')]:
    del sys.modules[_mod]

import sqlite3
import pandas as pd

from financial_sentiment.config import DB_PATH
from financial_sentiment import Backtester  # sanity check
print(f"financial_sentiment loaded from: {__import__('financial_sentiment').__file__}")


financial_sentiment loaded from: /Users/badremhiouah/Desktop/Preparation Entretiens/Applied AI project 2/src/financial_sentiment/__init__.py


## News Table

In [68]:
conn = sqlite3.connect(DB_PATH)
news = pd.read_sql("SELECT * FROM news", conn)
conn.close()
news.head(2)

,id,symbol,date,headline,text,source,url,sentiment_label,confidence,finbert_score,llm_label,llm_positive,llm_negative,llm_neutral,llm_score
0,1,AAPL,2026-07-29,Apple Could Swing $170 Billion In Value After ...,<p>Earnings are back in the driver’s seat on T...,OpenBB,https://www.benzinga.com/markets/options/26/07...,neutral,0.0,0.0,None,None,None,None,None
1,2,AAPL,2026-07-29,"Apple Q3 Preview: Stock Hits All-Time Highs, W...",<p>Technology giant <strong>Apple Inc</strong>...,OpenBB,https://www.benzinga.com/trading-ideas/preview...,neutral,0.0,0.0,None,None,None,None,None


## Prices Table

In [69]:
conn = sqlite3.connect(DB_PATH)
prices = pd.read_sql("SELECT * FROM prices", conn)
conn.close()
prices

,id,symbol,date,open,high,low,close,volume,daily_return,weekly_return,monthly_return
0,1,AAPL,2026-07-01,293.44,296.59,289.20,294.38,50164232.0,NaN,NaN,NaN
1,2,AAPL,2026-07-02,294.12,309.42,293.68,308.63,75400626.0,0.048407,NaN,NaN
2,3,AAPL,2026-07-06,307.36,314.20,307.00,312.66,53590000.0,0.013058,NaN,NaN
3,4,AAPL,2026-07-07,315.29,315.48,310.15,310.66,42490002.0,-0.006397,NaN,NaN
4,5,AAPL,2026-07-08,311.91,314.82,307.05,313.39,41323500.0,0.008788,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
289,446,META,2026-08-10,599.13,608.21,592.03,594.92,14967328.0,0.004763,0.103686,NaN
290,447,META,2026-08-11,598.00,612.43,593.40,599.12,13146900.0,0.007060,0.076180,NaN
291,448,META,2026-08-12,601.07,604.50,578.21,578.85,16186915.0,-0.033833,-0.019297,NaN
292,449,META,2026-08-13,580.71,595.85,579.39,594.97,11164200.0,0.027848,0.011957,-0.029270


## Sentiment Scores (FinBERT)

In [70]:
conn = sqlite3.connect(DB_PATH)
news_sentiment = pd.read_sql("""
    SELECT symbol, date, headline, sentiment_label, confidence, finbert_score
    FROM news
    WHERE finbert_score IS NOT NULL
""", conn)
conn.close()
news_sentiment[news_sentiment['finbert_score'] != 0]

,symbol,date,headline,sentiment_label,confidence,finbert_score
5,AAPL,2026-07-29,SK Hynix Just Had Its Best Quarter Ever: The S...,negative,-0.974915,-0.974915
6,AAPL,2026-07-29,"Corning Landed Apple, Meta, Nvidia and Amazon ...",positive,0.851358,0.851358
8,AAPL,2026-07-29,Apple's Incoming CEO John Ternus Signals More ...,positive,0.380353,0.380353
9,AAPL,2026-08-26,Apple's 'Surprise and Shine' iPhone 18 Teaser ...,positive,0.695088,0.695088
18,AAPL,2026-07-22,What's Going on With Apple Stock Wednesday?,negative,-0.551996,-0.551996
...,...,...,...,...,...,...
1242,META,2026-08-27,Meta to transform social media platforms for t...,positive,0.812790,0.812790
1246,META,2026-08-26,Meta's settlement has mostly put its legal iss...,negative,0.564149,-0.564149
1253,META,2026-08-21,No more doom scrolling or Instagram Stories? A...,negative,0.818745,-0.818745
1264,META,2026-08-18,Former Assistant Attorney General Jonathan Kan...,negative,0.496372,-0.496372


## Correlation: Sentiment vs Returns

Ignore days where sentiment score is 0. Compute correlation between average daily sentiment and daily, weekly, and monthly returns.

In [71]:
import math
from collections import defaultdict

# Aggregate sentiment by symbol + date
daily_sentiment = (
    news_sentiment.groupby(["symbol", "date"])["finbert_score"]
    .mean()
    .reset_index()
)

sentiment_map = {}
for _, row in daily_sentiment.iterrows():
    sentiment_map[(row["symbol"], row["date"])] = row["finbert_score"]

by_symbol = defaultdict(list)
for _, row in prices.iterrows():
    by_symbol[row["symbol"]].append(row)

def corr(x, y):
    n = len(x)
    if n < 2:
        return None
    mx = sum(x) / n
    my = sum(y) / n
    num = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
    dx = math.sqrt(sum((xi - mx) ** 2 for xi in x))
    dy = math.sqrt(sum((yi - my) ** 2 for yi in y))
    if dx == 0 or dy == 0:
        return None
    return num / (dx * dy)

print("Correlation by symbol (ignoring sentiment_score == 0):")
print(f"{'Symbol':<8} {'Daily':>10} {'Weekly':>10} {'Monthly':>10} {'N':>5}")
print("-" * 45)

for symbol, rows in sorted(by_symbol.items()):
    rows.sort(key=lambda x: x["date"])
    daily_x, daily_y = [], []
    weekly_x, weekly_y = [], []
    monthly_x, monthly_y = [], []

    for row in rows:
        score = sentiment_map.get((symbol, row["date"]))
        if score is None or score == 0:
            continue
        daily_r = row.get("daily_return")
        weekly_r = row.get("weekly_return")
        monthly_r = row.get("monthly_return")
        if daily_r is not None and not (isinstance(daily_r, float) and math.isnan(daily_r)):
            daily_x.append(score)
            daily_y.append(daily_r)
        if weekly_r is not None and not (isinstance(weekly_r, float) and math.isnan(weekly_r)):
            weekly_x.append(score)
            weekly_y.append(weekly_r)
        if monthly_r is not None and not (isinstance(monthly_r, float) and math.isnan(monthly_r)):
            monthly_x.append(score)
            monthly_y.append(monthly_r)

    d = corr(daily_x, daily_y)
    w = corr(weekly_x, weekly_y)
    m = corr(monthly_x, monthly_y)

    d_str = f"{d:>10.3f}" if d is not None else "N/A"
    w_str = f"{w:>10.3f}" if w is not None else "N/A"
    m_str = f"{m:>10.3f}" if m is not None else "N/A"
    n = len(daily_x)

    print(f"{symbol:<8} {d_str:>10} {w_str:>10} {m_str:>10} {n:>5}")

Correlation by symbol (ignoring sentiment_score == 0):
Symbol        Daily     Weekly    Monthly     N
---------------------------------------------
AAPL          0.167      0.221      0.068    40
AMZN            N/A        N/A        N/A     1
GOOGL         0.024     -0.333      0.893    24
META         -0.029      0.703        N/A     5
MSFT          0.216      0.163      0.199    41
NVDA          0.012      0.200      0.100    18
TSLA            N/A        N/A        N/A     0


## Correlation & Distribution by Equity

Each row corresponds to one equity. Left: bucket heatmap (15 equally spaced buckets) showing the percentage of data points per sentiment/return bin (sums to 100%), with % text in each cell. Log returns are capped at 5th/95th percentiles. Right: correlation bar chart for all return types. Use the dropdown to switch between daily, weekly, and monthly.

In [72]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

conn = sqlite3.connect(DB_PATH)
prices_df = pd.read_sql("SELECT * FROM prices", conn)
conn.close()
prices_df["date"] = pd.to_datetime(prices_df['date'], format='mixed', utc=True, errors='coerce').dt.tz_localize(None)
news_sentiment["date"] = pd.to_datetime(news_sentiment['date'], format='mixed', utc=True, errors='coerce').dt.tz_localize(None)

daily_sentiment = (
    news_sentiment.groupby(["symbol", "date"])["finbert_score"]
    .mean()
    .reset_index()
)

merged = prices_df.merge(daily_sentiment, on=["symbol", "date"], how="inner")
merged = merged[merged["finbert_score"] != 0].copy()

for rtype in ["daily_return", "weekly_return", "monthly_return"]:
    col = f"log_{rtype}"
    merged[col] = np.where(
        (merged[rtype].notna()) & (merged[rtype] > -1),
        np.log1p(merged[rtype]),
        np.nan,
    )

MIN_SENTIMENT_N = 5  # ignore symbols with fewer sentiment observations than this

sentiment_counts = news_sentiment.groupby("symbol").size()
qualifying_symbols = sentiment_counts[sentiment_counts >= MIN_SENTIMENT_N].index.tolist()

symbols = sorted(merged["symbol"].unique().tolist())
symbols = [s for s in symbols if s in news_sentiment["symbol"].values and s in qualifying_symbols]
return_cols = ["log_daily_return", "log_weekly_return", "log_monthly_return"]
return_labels = ["Daily", "Weekly", "Monthly"]

n_buckets_x = 12
n_buckets_y = 12

fig = make_subplots(
    rows=len(symbols),
    cols=2,
    subplot_titles=[title for pair in zip([f"{s} - Distribution" for s in symbols], [f"{s} - Correlation" for s in symbols]) for title in pair],
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
    specs=[[{"type": "heatmap"}, {"type": "bar"}] for _ in symbols],
)

corr_bars = []
for symbol in symbols:
    symbol_data = merged[merged["symbol"] == symbol].copy()
    corr_vals = []
    for rcol, rlabel in zip(return_cols, return_labels):
        col_data = symbol_data[[rcol, "finbert_score"]].dropna()
        if len(col_data) < 2:
            corr_vals.append(0)
            continue
        x = col_data["finbert_score"].values
        y = col_data[rcol].values
        mx, my = x.mean(), y.mean()
        num = ((x - mx) * (y - my)).sum()
        dx, dy = np.sqrt(((x - mx) ** 2).sum()), np.sqrt(((y - my) ** 2).sum())
        corr_vals.append(num / (dx * dy) if dx and dy else 0)
    corr_bars.append(corr_vals)

visibility_daily = []
visibility_weekly = []
visibility_monthly = []

for idx, symbol in enumerate(symbols, start=1):
    symbol_data = merged[merged["symbol"] == symbol].copy()

    for rcol, rlabel in zip(return_cols, return_labels):
        col_data = symbol_data[[rcol, "finbert_score"]].dropna()
        if len(col_data) < 4:
            continue

        vals = col_data[rcol].values
        lo, hi = np.percentile(vals, 5), np.percentile(vals, 95)
        if hi <= lo:
            lo, hi = vals.min(), vals.max()
            if hi <= lo:
                hi = lo + 1e-6
        col_data[rcol] = col_data[rcol].clip(lo, hi)
        score_min, score_max = col_data["finbert_score"].min(), col_data["finbert_score"].max()
        if score_max <= score_min:
            score_max = score_min + 1e-6

        score_bin_edges = np.linspace(score_min, score_max, n_buckets_x + 1)
        ret_bin_edges = np.linspace(lo, hi, n_buckets_y + 1)
        x = col_data["finbert_score"].values
        y = col_data[rcol].values
        mask = ~(np.isnan(x) | np.isnan(y))
        x, y = x[mask], y[mask]
        H, _, _ = np.histogram2d(x, y, bins=[score_bin_edges, ret_bin_edges])
        total = H.sum()
        hist2d = (H / total * 100).T if total > 0 else H.T

        score_labels = [f"{score_bin_edges[i]:.3f} to {score_bin_edges[i+1]:.3f}" for i in range(n_buckets_y)]
        ret_labels = [f"{ret_bin_edges[i]:.4f} to {ret_bin_edges[i+1]:.4f}" for i in range(n_buckets_y)]

        z_text = [[f"{val:.1f}%" for val in row] for row in hist2d]

        visible = (rlabel == "Daily")
        fig.add_trace(
            go.Heatmap(
                z=hist2d,
                x=score_labels,
                y=ret_labels,
                colorscale="Blues",
                showscale=False,
                text=z_text,
                texttemplate="%{text}",
                textfont={"size": 8},
                visible=visible,
                hovertemplate="Score bucket: %{x}<br>Return bucket: %{y}<br>Percentage: %{z:.1f}%<extra></extra>",
            ),
            row=idx,
            col=1,
        )

        if rlabel == "Daily":
            visibility_daily.append(True)
            visibility_weekly.append(False)
            visibility_monthly.append(False)
        elif rlabel == "Weekly":
            visibility_daily.append(False)
            visibility_weekly.append(True)
            visibility_monthly.append(False)
        else:
            visibility_daily.append(False)
            visibility_weekly.append(False)
            visibility_monthly.append(True)

    fig.add_trace(
        go.Bar(
            x=return_labels,
            y=corr_bars[idx - 1],
            marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"],
            text=[f"{v:.3f}" for v in corr_bars[idx - 1]],
            textposition="outside",
            name=f"{symbol} correlation",
        ),
        row=idx,
        col=2,
    )
    visibility_daily.append(True)
    visibility_weekly.append(True)
    visibility_monthly.append(True)

fig.update_layout(
    title="Sentiment vs Return: Distribution & Correlation by Equity",
    height=450 * len(symbols),
    updatemenus=[
        dict(
            buttons=[
                dict(label="Daily", method="update", args=[{"visible": visibility_daily}]),
                dict(label="Weekly", method="update", args=[{"visible": visibility_weekly}]),
                dict(label="Monthly", method="update", args=[{"visible": visibility_monthly}]),
            ],
            direction="down",
            showactive=True,
            x=0.27,
            y=1.15,
            xanchor="left",
            yanchor="top",
        )
    ],
)
fig.update_xaxes(title_text="FinBERT Score Bucket", row=len(symbols), col=1)
fig.update_yaxes(title_text="Log Return Bucket", row=len(symbols), col=1)
fig.update_xaxes(title_text="Return Type", row=len(symbols), col=2)
fig.update_yaxes(title_text="Correlation", row=len(symbols), col=2, range=[-1, 1])
fig











## Interactive Plot: Price + Sentiment by Equity

In [73]:
import plotly.graph_objects as go

conn = sqlite3.connect(DB_PATH)
prices_df = pd.read_sql("SELECT * FROM prices", conn)
conn.close()
prices_df["date"] = pd.to_datetime(prices_df['date'], format='mixed', utc=True, errors='coerce').dt.tz_localize(None)
news_sentiment["date"] = pd.to_datetime(news_sentiment['date'], format='mixed', utc=True, errors='coerce').dt.tz_localize(None)

# Aggregate sentiment by symbol + date, keeping headlines
daily_sentiment = (
    news_sentiment.groupby(["symbol", "date"])
    .agg(
        finbert_score=("finbert_score", "mean"),
        headline=("headline", lambda titles: ", ".join(titles.astype(str))),
    )
    .reset_index()
)

# Derive sentiment label from mean score
daily_sentiment["sentiment_label"] = daily_sentiment["finbert_score"].apply(
    lambda s: "positive" if s > 0.2 else "negative" if s < -0.2 else "neutral"
)

# Merge prices with aggregated sentiment
merged = prices_df.merge(daily_sentiment, on=["symbol", "date"], how="left")

fig = go.Figure()

colors = {"positive": "green", "negative": "red", "neutral": "gray"}
all_symbols = sorted(prices_df["symbol"].unique().tolist())
all_symbols = [s for s in all_symbols if s in news_sentiment["symbol"].values]
# Drop symbols with too few sentiment observations (not statistically meaningful)
MIN_SENTIMENT_N = 5
sentiment_counts = news_sentiment.groupby("symbol").size()
qualifying = sentiment_counts[sentiment_counts >= MIN_SENTIMENT_N].index.tolist()
all_symbols = [s for s in all_symbols if s in qualifying]
print(f"Symbols with >= {MIN_SENTIMENT_N} sentiment observations: {all_symbols}")

for symbol in all_symbols:
    stock_data = merged[merged["symbol"] == symbol].sort_values("date")

    fig.add_trace(
        go.Scatter(
            x=stock_data["date"].to_numpy(),
            y=stock_data["close"],
            mode="lines",
            name=f"{symbol}",
            line=dict(width=2),
        )
    )

    # Add sentiment markers only for this symbol
    has_sentiment = stock_data["finbert_score"].notna()
    if has_sentiment.any():
        sentiment_data = stock_data[has_sentiment]
        fig.add_trace(
            go.Scatter(
                x=sentiment_data["date"].to_numpy(),
                y=sentiment_data["close"],
                mode="markers",
                name=f"{symbol} sentiment",
                marker=dict(
                    size=(sentiment_data["finbert_score"].abs() * 30).clip(6, 25),
                    color=[colors.get(l, "gray") for l in sentiment_data["sentiment_label"]],
                    line=dict(width=1, color="black"),
                    opacity=0.9,
                ),
                text=sentiment_data["headline"],
                hovertemplate=(
                    "%{text}<br>"
                    "Date: %{x|%Y-%m-%d}<br>"
                    "Close: %{y:.2f}<br>"
                    "Sentiment: %{customdata[0]}<br>"
                    "Score: %{customdata[1]:.2f}"
                ),
                customdata=sentiment_data[["sentiment_label", "finbert_score"]].values,
            )
        )

fig.update_layout(
    title="Price + News Sentiment by Equity",
    xaxis_title="Date",
    yaxis_title="Close Price",
    hovermode="x unified",
    height=600,
)
fig



Symbols with >= 5 sentiment observations: ['AAPL', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']


## Quick Summary

In [74]:
print(f"News rows: {len(news)}")
print(f"Prices rows: {len(prices)}")
print("News symbols:", news["symbol"].unique().tolist() if len(news) else [])
print("Price symbols:", prices["symbol"].unique().tolist() if len(prices) else [])
print(f"FinBERT scores: {news_sentiment['finbert_score'].notna().sum()}")

News rows: 1268
Prices rows: 294
News symbols: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'TSLA', 'META']
Price symbols: ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'NVDA', 'TSLA', 'META']
FinBERT scores: 1268


## Trading Strategy Backtest Demo

Run the sentiment-threshold strategy on a randomly chosen equity that already exists in the database, then visualize the evolution of `virtual_capital` along with the number of open long/short trades at each tick.

- `capital` is the available cash (open trades are funded from it).
- `virtual_capital` = `capital` + liquidation value of all open trades at the current price.
- On the final trading day all open trades are liquidated, so the last `capital` equals the last `virtual_capital`.

In [75]:
import random
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from financial_sentiment import (
    Backtester, Portfolio, SentimentThresholdStrategy,
    InsufficientSignalsError,
)

# 1. Pick a random equity that already exists in the database.
conn = sqlite3.connect(DB_PATH)
available_equities = sorted(
    r[0] for r in conn.execute("SELECT DISTINCT symbol FROM prices")
)
conn.close()
random.seed()  # uncomment to fix: random.seed(42)
equity = random.choice(available_equities)
print(f"Selected equity: {equity}")

# 2. Instantiate the strategy and portfolio with example parameters.
portfolio = Portfolio(
    initial_capital=100,
    trade_size=20,
    stop_loss=-0.05,
    stop_win=0.10,
)
strategy = SentimentThresholdStrategy(
    buy_threshold=0.2,
    sell_threshold=-0.2,
)

# 3. Run the backtest (raises InsufficientSignalsError if < 5 signals).
try:
    result = Backtester(portfolio, strategy).run(equity)
except InsufficientSignalsError as e:
    raise SystemExit(f"Strategy invalid for {equity}: {e}")

print(f"Signals: {Backtester(portfolio, strategy).count_signals(equity)}")
print(f"Closed trades: {len(portfolio.closed_trades)}")
print(f"Final capital: {portfolio.capital:.2f}  (initial: {portfolio.initial_capital})")

# 4. Display the enriched DataFrame (head + tail).
display_cols = ["date", "close", "sentiment_score",
               "capital", "virtual_capital", "open_long", "open_short"]
print("\nEnriched DataFrame (head):")
print(result[display_cols].head().to_string(index=False))
print("\nEnriched DataFrame (tail):")
print(result[display_cols].tail().to_string(index=False))

# 5. Plot: virtual_capital evolution + open_long/open_short as visual encoding.
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.7, 0.3],
    subplot_titles=(
        f"{equity}: Virtual Capital Over Time",
        "Number of Open Trades (long / short)",
    ),
)

# --- Row 1: virtual capital as the main trace, with initial capital as reference
fig.add_trace(
    go.Scatter(
        x=result["date"],
        y=result["virtual_capital"],
        mode="lines+markers",
        name="virtual_capital",
        line=dict(color="#1f77b4", width=2),
        marker=dict(
            size=8,
            color=result["open_long"] - result["open_short"],  # net exposure sign
            colorscale="RdBu",
            cmin=-3, cmax=3,
            showscale=False,
            line=dict(width=1, color="black"),
        ),
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Virtual capital: %{y:.2f}<br>"
            "Open long: %{customdata[0]}<br>"
            "Open short: %{customdata[1]}<extra></extra>"
        ),
        customdata=result[["open_long", "open_short"]].values,
    ),
    row=1, col=1,
)

fig.add_hline(
    y=portfolio.initial_capital,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"initial capital = {portfolio.initial_capital}",
    annotation_position="bottom right",
    row=1, col=1,
)

# --- Row 2: stacked-area style bars for long / short counts
fig.add_trace(
    go.Bar(
        x=result["date"], y=result["open_long"],
        name="open_long", marker_color="#2ca02c", opacity=0.85,
    ),
    row=2, col=1,
)
fig.add_trace(
    go.Bar(
        x=result["date"], y=-result["open_short"],
        name="open_short", marker_color="#d62728", opacity=0.85,
    ),
    row=2, col=1,
)

fig.update_layout(
    barmode="overlay",
    title=f"{equity} Backtest: virtual_capital and open trade counts",
    hovermode="x unified",
    height=650,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_yaxes(title_text="Virtual capital ($)", row=1, col=1)
fig.update_yaxes(title_text="Open long (+) / short (-)", row=2, col=1, zeroline=True)
fig.update_xaxes(title_text="Date", row=2, col=1)
fig


Selected equity: MSFT
Signals: 9
Closed trades: 9
Final capital: 105.99  (initial: 100)

Enriched DataFrame (head):
      date  close  sentiment_score  capital  virtual_capital  open_long  open_short
2026-07-01 384.28         0.205519     80.0       100.000000          1           0
2026-07-02 390.49        -0.173784     80.0       100.323202          1           0
2026-07-06 386.74        -0.658097     60.0       100.128032          1           1
2026-07-07 388.84        -0.075416     60.0       100.128727          1           1
2026-07-08 383.34         0.066598     60.0       100.126906          1           1

Enriched DataFrame (tail):
      date  close  sentiment_score    capital  virtual_capital  open_long  open_short
2026-08-24 487.31        -0.097908  47.471186       107.085090          1           2
2026-08-25 491.71         0.086554  47.471186       106.901021          1           2
2026-08-26 496.37        -0.060480  47.471186       106.706075          1           2
2026-08-

/Users/badremhiouah/miniconda3/lib/python3.10/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



## FinBERT vs LLM Sentiment Comparison

For each headline that has both a FinBERT score and an LLM (Groq) score, plot them against each other to compare the two sentiment models.

The LLM scores are produced by `GroqSentimentScorer` and stored in the `llm_score` column. To populate them, run `python scripts/score_llm.py --limit <N>` (or unset `--limit` for all rows).

In [76]:
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sqlite3
import pandas as pd

# Load rows that have BOTH scores
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("""
    SELECT symbol, date, headline,
           finbert_score, sentiment_label AS finbert_label,
           llm_score, llm_label,
           llm_positive, llm_negative, llm_neutral
    FROM news
    WHERE finbert_score IS NOT NULL
      AND finbert_score != 0
      AND llm_score IS NOT NULL
""", conn)
conn.close()
print(f"Rows with both FinBERT and LLM scores: {len(df)}")
if len(df):
    print(df.head(3).to_string(index=False))

# Scatter plot
if len(df) == 0:
    print("\nNo data to plot yet. Run `python scripts/score_llm.py --limit 200` to backfill LLM scores.")
else:
    # Disagreement heat: where FinBERT and LLM disagree on the sign of the score
    df["agree"] = np.sign(df["finbert_score"]) == np.sign(df["llm_score"])

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("FinBERT vs LLM sentiment score", "Disagreement by symbol"),
        column_widths=[0.7, 0.3],
    )

    # Main scatter
    fig.add_trace(go.Scatter(
        x=df["finbert_score"], y=df["llm_score"],
        mode="markers",
        marker=dict(
            size=6, opacity=0.5,
            color=df["agree"].map({True: "#2ca02c", False: "#d62728"}),
        ),
        text=df["headline"],
        customdata=df[["symbol", "finbert_label", "llm_label"]].values,
        hovertemplate=("<b>%{customdata[0]}</b> %{x|%Y-%m-%d}<br>"
                       "FinBERT: %{x:+.2f} (%{customdata[1]})<br>"
                       "LLM: %{y:+.2f} (%{customdata[2]})<br>"
                       "<i>%{text}</i><extra></extra>"),
    ), row=1, col=1)

    # y=x reference line
    fig.add_trace(go.Scatter(
        x=[-1, 1], y=[-1, 1], mode="lines",
        line=dict(color="gray", dash="dash"),
        showlegend=False, hoverinfo="skip",
    ), row=1, col=1)

    fig.update_xaxes(title_text="FinBERT score", range=[-1.05, 1.05], row=1, col=1)
    fig.update_yaxes(title_text="LLM (Groq) score", range=[-1.05, 1.05], row=1, col=1)

    # Disagreement by symbol
    grp = df.groupby("symbol").agg(
        n=("agree", "size"),
        agree=("agree", "sum"),
    ).reset_index()
    grp["pct_agree"] = 100 * grp["agree"] / grp["n"]
    grp = grp.sort_values("pct_agree")
    fig.add_trace(go.Bar(
        x=grp["pct_agree"], y=grp["symbol"],
        orientation="h",
        marker_color="#1f77b4",
        text=[f"{p:.0f}% (n={n})" for p, n in zip(grp["pct_agree"], grp["n"])],
        textposition="outside",
        showlegend=False,
    ), row=1, col=2)
    fig.update_xaxes(title_text="% sign-agreement", range=[0, 105], row=1, col=2)
    fig.update_yaxes(title_text="", row=1, col=2)

    fig.update_layout(
        title="FinBERT vs LLM sentiment: per-headline comparison",
        height=600,
    )
    fig

    # Overall correlation
    from financial_sentiment import pearson
    r = pearson(df["finbert_score"].tolist(), df["llm_score"].tolist())
    print(f"\nPearson correlation (FinBERT vs LLM): {r:.3f}  (n={len(df)})")


Rows with both FinBERT and LLM scores: 0

No data to plot yet. Run `python scripts/score_llm.py --limit 200` to backfill LLM scores.


## Strategy Parameter Optimisation (1-month training window)

Search the (stop_loss, stop_win, buy_threshold, sell_threshold) space on a 1-month window, optimising total P&L (final capital - initial capital).

Two-stage search: a coarse grid, then a refinement grid around each of the top-K coarse candidates. Configurable via `StrategyOptimizer(coarse_grid=..., top_k=..., refine_factor=..., refine_step_count=...)`.

In [77]:
import time
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from financial_sentiment import (
    StrategyOptimizer, Backtester, Portfolio, SentimentThresholdStrategy,
    InsufficientSignalsError,
)

# --- pick a random equity that has at least 1 month of recent price data ---
conn = sqlite3.connect(DB_PATH)
rows = conn.execute("""
    SELECT symbol, MIN(date), MAX(date), COUNT(*)
    FROM prices
    GROUP BY symbol
    HAVING COUNT(*) >= 18
""").fetchall()
conn.close()
candidates = [r[0] for r in rows if r[2] >= "2026-08-15"]  # at least ~2 weeks
random.seed(42)
equity = random.choice(candidates)
print(f"Optimising equity: {equity}")

# --- 1-month training window: the last ~30 days of available prices ---
conn = sqlite3.connect(DB_PATH)
date_min, date_max = conn.execute(
    "SELECT MIN(date), MAX(date) FROM prices WHERE symbol = ?", (equity,)
).fetchone()
conn.close()
end_dt = datetime.strptime(date_max, "%Y-%m-%d")
start_dt = datetime.strptime(date_min, "%Y-%m-%d")
# Use the last 30 calendar days (or all of it if shorter)
if (end_dt - start_dt).days > 30:
    start_dt = end_dt - pd.Timedelta(days=30)
print(f"Training window: {start_dt.date()} -> {end_dt.date()}")

# --- run the coarse-then-refine search ---
t0 = time.time()
result = StrategyOptimizer().optimize(equity, start_dt, end_dt)
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s. Trials: {len(result.all_trials)}")

print("\nBest parameters:")
for k, v in result.best.params.items():
    print(f"  {k:>15}: {v}")
print(f"\nBest P&L        : {result.best.pnl:+.2f}")
print(f"Final capital    : {result.best.final_capital:.2f}  (initial 100)")
print(f"Signals          : {result.best.n_signals}")
print(f"Closed trades    : {result.best.n_trades}")

# --- top-10 leaderboard as a DataFrame ---
top = pd.DataFrame([t.__dict__ for t in result.all_trials[:10]])
print("\nTop-10 trials:")
print(top.to_string(index=False))

# --- run the backtest with the best params and plot virtual_capital ---
portfolio = Portfolio(
    initial_capital=100, trade_size=20,
    stop_loss=result.best.stop_loss, stop_win=result.best.stop_win,
)
strategy = SentimentThresholdStrategy(
    buy_threshold=result.best.buy_threshold,
    sell_threshold=result.best.sell_threshold,
)
df = Backtester(portfolio, strategy).run(equity)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    row_heights=[0.7, 0.3],
    subplot_titles=(
        f"{equity}: Virtual capital with optimised parameters",
        "Number of open trades",
    ),
)
fig.add_trace(go.Scatter(
    x=df["date"], y=df["virtual_capital"], mode="lines+markers",
    name="virtual_capital", line=dict(color="#1f77b4", width=2),
    marker=dict(size=6, color=df["open_long"] - df["open_short"],
                colorscale="RdBu", cmin=-3, cmax=3, showscale=False,
                line=dict(width=1, color="black")),
    customdata=df[["open_long", "open_short"]].values,
    hovertemplate=("Date %{x|%Y-%m-%d}<br>Virtual capital %{y:.2f}"
                  "<br>Open L: %{customdata[0]}  Open S: %{customdata[1]}<extra></extra>"),
), row=1, col=1)
fig.add_hline(y=100, line_dash="dash", line_color="gray",
              annotation_text="initial = 100", annotation_position="bottom right",
              row=1, col=1)
fig.add_trace(go.Bar(x=df["date"], y=df["open_long"], name="open_long",
                     marker_color="#2ca02c", opacity=0.85), row=2, col=1)
fig.add_trace(go.Bar(x=df["date"], y=-df["open_short"], name="open_short",
                     marker_color="#d62728", opacity=0.85), row=2, col=1)
fig.update_layout(barmode="overlay", height=650, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02,
                              xanchor="right", x=1))
fig.update_yaxes(title_text="Virtual capital ($)", row=1, col=1)
fig.update_yaxes(title_text="Open long (+) / short (-)", row=2, col=1, zeroline=True)
fig.update_xaxes(title_text="Date", row=2, col=1)
fig

# --- heatmap: P&L across (stop_loss, stop_win) at the best thresholds ---
import numpy as np
best_t = result.best
sl_vals = sorted({t.stop_loss for t in result.all_trials})
sw_vals = sorted({t.stop_win for t in result.all_trials})
grid = np.full((len(sl_vals), len(sw_vals)), np.nan)
for t in result.all_trials:
    i = sl_vals.index(t.stop_loss)
    j = sw_vals.index(t.stop_win)
    if t.buy_threshold == best_t.buy_threshold and t.sell_threshold == best_t.sell_threshold:
        grid[i, j] = t.pnl
fig2 = go.Figure(go.Heatmap(
    z=grid, x=sw_vals, y=sl_vals, colorscale="RdYlGn",
    zmid=0, text=[[f"{v:+.1f}" if not np.isnan(v) else "" for v in row] for row in grid],
    texttemplate="%{text}", hovertemplate="stop_win=%{x}<br>stop_loss=%{y}<br>P&L=%{z:+.2f}<extra></extra>",
))
fig2.update_layout(
    title=f"{equity}: P&L across (stop_loss, stop_win) at best buy={best_t.buy_threshold}, sell={best_t.sell_threshold}",
    xaxis_title="stop_win", yaxis_title="stop_loss", height=400,
)
fig2


Optimising equity: NVDA
Training window: 2026-07-29 -> 2026-08-28

Done in 56.1s. Trials: 1956

Best parameters:
        stop_loss: -0.15
         stop_win: 0.025
    buy_threshold: 0.2
   sell_threshold: -0.1

Best P&L        : +7.34
Final capital    : 107.34  (initial 100)
Signals          : 16
Closed trades    : 16

Top-10 trials:
 stop_loss  stop_win  buy_threshold  sell_threshold      pnl  final_capital  n_signals  n_trades
    -0.150     0.025           0.20           -0.10 7.337943     107.337943         16        16
    -0.150     0.025           0.20           -0.05 7.337943     107.337943         16        16
    -0.125     0.025           0.20           -0.10 7.337943     107.337943         16        16
    -0.125     0.025           0.20           -0.05 7.337943     107.337943         16        16
    -0.100     0.025           0.20           -0.10 7.337943     107.337943         16        16
    -0.100     0.025           0.20           -0.05 7.337943     107.337943       

/Users/badremhiouah/miniconda3/lib/python3.10/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result

